# QUEST 3

_The hypothesis was that the page would create peer pressure and the students would start working on the labs earlier_

In [3]:
import pandas as pd
import sqlite3

## 1) create a connection to the database using the library sqlite3

In [4]:
try:

    connection = sqlite3.connect('../data/checking-logs.sqlite')
    cursor = connection.cursor()
except Exception:
    print("The connection is failed :(")

## 2) get the schema of the table test

In [5]:
pd.io.sql.read_sql("PRAGMA table_info(test);", connection)

,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,uid,TEXT,0,None,0
2,2,labname,TEXT,0,None,0
3,3,first_commit_ts,TIMESTAMP,0,None,0
4,4,first_view_ts,TIMESTAMP,0,None,0


## 3) get only the first 10 rows of the table test to check what the table looks like

In [6]:
pd.io.sql.read_sql("select * from test limit 10", connection)

,index,uid,labname,first_commit_ts,first_view_ts
0,3,user_17,project1,2020-04-18 07:56:45,2020-04-18 10:56:55
1,4,user_30,laba04,2020-04-18 13:36:53,2020-04-17 22:46:26
2,7,user_30,laba04s,2020-04-18 14:51:37,2020-04-17 22:46:26
3,8,user_14,laba04,2020-04-18 15:14:00,2020-04-18 10:53:52
4,11,user_14,laba04s,2020-04-18 22:30:30,2020-04-18 10:53:52
5,18,user_19,laba04,2020-04-20 19:05:01,2020-04-21 20:30:38
6,19,user_25,laba04,2020-04-20 19:16:50,2020-05-09 23:54:54
7,20,user_21,laba04,2020-04-21 17:48:00,2020-04-22 22:40:36
8,21,user_30,project1,2020-04-22 12:36:24,2020-04-17 22:46:26
9,23,user_21,laba04s,2020-04-22 20:09:21,2020-04-22 22:40:36


## 3) find among all the users the minimum value of the delta between the first commit of the user and the deadline of the corresponding lab using only one query

* do this by joining the table with the table deadlines
* the difference should be displayed in hours
* do not take the lab _’project1’_ into account, it has longer deadlines and will be an outlier
* the value should be stored in the dataframe df_min with the corresponding uid

Выведем сначала таблицу с дельтой:

In [7]:
query = """
select t.uid, d.labs, t.first_commit_ts, datetime(d.deadlines, 'unixepoch'), 
cast((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 as integer) as delta from test t
left join deadlines d on t.labname = d.labs 
where t.labname != 'project1'
order by t.uid
"""
pd.io.sql.read_sql(query, connection)

,uid,labs,first_commit_ts,"datetime(d.deadlines, 'unixepoch')",delta
0,user_1,laba04,2020-04-26 17:06:18,2020-04-26 23:59:59,-6
1,user_1,laba04s,2020-04-26 17:12:11,2020-04-26 23:59:59,-6
2,user_1,laba05,2020-05-02 19:15:18,2020-05-03 23:59:59,-28
3,user_1,laba06,2020-05-17 16:26:35,2020-05-24 23:59:59,-175
4,user_1,laba06s,2020-05-20 12:23:37,2020-05-24 23:59:59,-107
5,user_10,laba04,2020-04-25 08:24:52,2020-04-26 23:59:59,-39
6,user_10,laba04s,2020-04-25 08:37:54,2020-04-26 23:59:59,-39
7,user_10,laba05,2020-05-01 19:27:26,2020-05-03 23:59:59,-52
8,user_10,laba06,2020-05-19 11:39:28,2020-05-24 23:59:59,-132
9,user_10,laba06s,2020-05-20 07:37:31,2020-05-24 23:59:59,-112


In [8]:
query_min = """
select uid, min(delta) from (
select t.uid, d.labs, t.first_commit_ts, datetime(d.deadlines, 'unixepoch'), 
cast((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 as integer) as delta from test t
left join deadlines d on t.labname = d.labs 
where t.labname != 'project1'
order by t.uid
)
"""
df_min = pd.io.sql.read_sql(query_min, connection, index_col="uid")
display(df_min)

,min(delta)
uid,
user_30,-202


## 4) do the same thing, but for the maximum, using only one query, the dataframe name is df_max

In [9]:
query_max = """
select uid, max(delta) from (
select t.uid, d.labs, t.first_commit_ts, datetime(d.deadlines, 'unixepoch'), 
cast((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 as integer) as delta from test t
left join deadlines d on t.labname = d.labs 
where t.labname != 'project1'
order by t.uid
)"""
df_max = pd.io.sql.read_sql(query_max, connection, index_col="uid")
display(df_max)

,max(delta)
uid,
user_25,-2


## 5) do the same thing but for the average, using only one query, this time your dataframe should not include the uid column, and the dataframe name is df_avg

In [10]:
query_avg = """
select avg(delta) from (
select t.uid, d.labs, t.first_commit_ts, datetime(d.deadlines, 'unixepoch'), 
cast((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 as integer) as delta from test t
left join deadlines d on t.labname = d.labs 
where t.labname != 'project1'
order by t.uid
)"""
df_avg = pd.io.sql.read_sql(query_avg, connection)
display(df_avg)

,avg(delta)
0,-89.125


## 6) we want to test the hypothesis that the users who visited the newsfeed just a few times have the lower delta between the first commit and the deadline. To do this, you need to calculate the correlation coefficient between the number of pageviews and the difference

* using only one query, create a table with the columns: uid, avg_diff, pageviews
* uid is the uids that exist in the test
* avg_diff is the average delta between the first commit and the lab deadline per user
* pageviews is the number of Newsfeed visits per user
* do not take the lab ’project1’ into account
* store it to the dataframe views_diff
* use the Pandas method corr() to calculate the correlation coefficient between the number of pageviews and the difference

In [11]:
query = """
select uid, avg(delta) as avg_diff, count(uid) as pageviews from (
	select t.uid, d.labs, t.first_commit_ts, datetime(d.deadlines, 'unixepoch'), 
	cast((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 as integer) as delta from test t
	left join deadlines d on t.labname = d.labs 
	where t.labname != 'project1'
	order by t.uid) 
group by uid
"""
views_diff = pd.io.sql.read_sql(query, connection, index_col="uid")

In [12]:
display(views_diff)

,avg_diff,pageviews
uid,,
user_1,-64.400000,5
user_10,-74.800000,5
user_14,-159.000000,3
user_17,-61.600000,5
user_18,-5.666667,3
user_19,-98.750000,4
user_21,-95.500000,4
user_25,-92.600000,5
user_28,-86.400000,5


### Подсчет коэффициента корреляции

Коэффициент корреляции обычно обозначается как  r  и принимает значения в диапазоне от -1 до 1:

•  r = 1 : Полная положительная линейная корреляция. Это означает, что при увеличении одной переменной другая также увеличивается.

•  r = -1 : Полная отрицательная линейная корреляция. Это означает, что при увеличении одной переменной другая уменьшается.

•  r = 0 : Нет линейной корреляции. Это означает, что между переменными нет линейной взаимосвязи.

_Сильная корреляция_: r ∊ [0.7; 1] V [-0.7; -1]

_Умеренная корреляция_: r ∊ (0.3; 0.7) V (-0.3; -0.7)

_Слабая корреляция_: r ∊ [0; 0.3] V [0; -0.3]

In [13]:
views_diff.corr(method="pearson")

,avg_diff,pageviews
avg_diff,1.000000,0.117685
pageviews,0.117685,1.000000


Коэффициент корреляции достаточно мал, то есть случайные величины avg_diff и pageviews несильно влияли друг на друга. Либо стоит исследовать не линейный тип взаимозвязи, например, коэффициент корреляции Спирмена:

In [14]:
views_diff.corr(method="spearman")

,avg_diff,pageviews
avg_diff,1.000000,0.316587
pageviews,0.316587,1.000000


Умеренной или сильной взаимосвязи также не обнаружено.

In [15]:
connection.close()